In [1]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import json
import os
import sys
from dataclasses import dataclass, field
from pathlib import Path
from typing import Literal

import optuna
import wandb
from dotenv import load_dotenv

sys.path.append(os.path.abspath("../.."))

import src.utils.run_optuna as op
from src.utils.optuna_objective import create_objective

### Config

In [2]:
load_dotenv(dotenv_path="../../.env")


@dataclass
class Config:
    # Data / CV
    model_name: str = "xgb"
    data_id: str = "057"
    n_folds: int = 5
    seed: int = 42
    fold_idx: int = 0

    # Optuna
    n_trials: int = 1
    direction: str = "maximize"
    sampler: str = "tpe"  # tpe / random
    pruner: str = "median"  # median / none

    # Initial params
    use_initial: Literal["never", "manual"] = "never"
    initial_param_sources: list[tuple[str, int]] = field(default_factory=list)   # (study_name, n_trial) 例: ("xgb-001", 1)

    # Storage
    storage: str = "sqlite:////home/hanse/kaggle/binary-bank/artifacts/optuna/optuna.db"

    # Option
    opts: dict = field(default_factory=dict)


cfg = Config()
cfg.initial_param_sources = [("lgbm-057", 16)]

# W&B
wandb_project = os.environ.get("COMPETITION_NAME")
wandb.login(key=os.environ.get("WANDB_API_KEY"))

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /home/hanse/.netrc
wandb: Currently logged in as: kaitookano (kaitookano-waseda-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

### Build & Run

In [3]:
# --- helper: initial params loader ---
def load_initial_params(sources: list[tuple[str, int]]) -> list[dict]:
    loaded = []
    for study, n_trial in sources:
        path = Path(f"../../artifacts/optuna/{study}/trl{n_trial}.json")
        with path.open("r") as f:
            params = json.load(f)["params"]
        loaded.append(params)
    return loaded


# sampler / pruner factory
def build_sampler(name, seed):
    if name == "tpe":
        return optuna.samplers.TPESampler(n_startup_trials=15, seed=seed)
    elif name == "random":
        return optuna.samplers.RandomSampler(seed=seed)
    else:
        raise ValueError(f"unknown sampler: {name}")


def build_pruner(name):
    if name == "median":
        return optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=1000)
    elif name == "none":
        return optuna.pruners.NopPruner()
    else:
        raise ValueError(f"unknown pruner: {name}")


objective = create_objective(
    cfg.model_name,
    cfg.data_id,
    seed=cfg.seed,
    n_folds=cfg.n_folds,
    fold_idx=cfg.fold_idx,
    wandb_project=wandb_project,
    study_name=f"{cfg.model_name}-{cfg.data_id}",
    opts=cfg.opts
)

sampler = build_sampler(cfg.sampler, cfg.seed)
pruner = build_pruner(cfg.pruner)

initial_params = None
if cfg.use_initial == "manual":
    initial_params = load_initial_params(cfg.initial_param_sources)

op.run_optuna_search(
    objective,
    n_trials=cfg.n_trials,
    direction=cfg.direction,
    study_name=f"{cfg.model_name}-{cfg.data_id}",
    storage=cfg.storage,
    sampler=sampler,
    pruner=pruner,
    initial_params=initial_params
)

[I 2025-10-17 23:57:10,060] Using an existing study with name 'xgb-057' instead of creating a new one.


[initial] none


  0%|          | 0/1 [00:00<?, ?it/s]

Fold Col: 5fold-s42
Free CPU Mem: 17.99 GB
Free GPU Mem: 6.66 GB

QuantileDMatrix Build Time: 00:00:09
[0]	train-auc:0.96011	valid-auc:0.95977
[100]	train-auc:0.97225	valid-auc:0.97170
[200]	train-auc:0.97354	valid-auc:0.97271
[300]	train-auc:0.97455	valid-auc:0.97350
[400]	train-auc:0.97531	valid-auc:0.97408
[500]	train-auc:0.97596	valid-auc:0.97454
[600]	train-auc:0.97654	valid-auc:0.97489
[700]	train-auc:0.97705	valid-auc:0.97520
[800]	train-auc:0.97751	valid-auc:0.97544
[900]	train-auc:0.97792	valid-auc:0.97561
[1000]	train-auc:0.97827	valid-auc:0.97576
[1100]	train-auc:0.97859	valid-auc:0.97588
[1200]	train-auc:0.97890	valid-auc:0.97599
[1300]	train-auc:0.97920	valid-auc:0.97608
[1400]	train-auc:0.97946	valid-auc:0.97615
[1500]	train-auc:0.97972	valid-auc:0.97622
[1600]	train-auc:0.97997	valid-auc:0.97628
[1700]	train-auc:0.98021	valid-auc:0.97634
[1800]	train-auc:0.98043	valid-auc:0.97639
[1900]	train-auc:0.98064	valid-auc:0.97643
[2000]	train-auc:0.98086	valid-auc:0.97647
[2100]

iter_f1,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▃▃▄▄▄▄▄▄▅▆▆▇▇▇▇▇▇▇▇▇████
train/f1/auc,▁▁▂▂▃▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇████
valid/f1/auc,▁▁▅▅▆▇▇▇▇▇██████████████████████████████
accuracy_f1,0.94746
auc_f1,0.97696
best_iter_f1,7671
iter_f1,8671
log_loss_f1,0.12365
mae_f1,0.07518
r2_f1,0.64242
rmse_f1,0.19477


[I 2025-10-18 00:04:21,271] Trial 8 finished with value: 0.9769642542228416 and parameters: {'learning_rate': 0.01, 'max_depth': 8, 'min_child_weight': 95.07143064099162, 'colsample_bytree': 0.592797576724562, 'subsample': 0.7394633936788146, 'reg_alpha': 0.0007482139197236472, 'reg_lambda': 0.000602521573620386}. Best is trial 5 with value: 0.9771938182552062.
✅ Message sent.
